<a href="https://colab.research.google.com/github/Marche1os/Z3-solver/blob/master/task4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install "z3-solver"
from z3 import *

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.7/31.7 MB 56.7 MB/s eta 0:00:00


**Раздел 1. Выражения, сорта, декларации**


Задача на сорт. Проверка возможности ситуации возникновения deadlock для двух потоков

In [3]:
Thread = DeclareSort('Thread')
Lock = DeclareSort('Lock')

holds = Function('holds', Thread, Lock, BoolSort())
waits = Function('waits', Thread, Lock, BoolSort())

t1, t2 = Consts('t1 t2', Thread)
l1, l2 = Consts('l1 l2', Lock)

s = Solver()

s.add(holds(t1, l1))
s.add(waits(t1, l2))

s.add(holds(t2, l2))
s.add(waits(t2, l1))

print(s.check())
print(s.model())

sat
[l2 = Lock!val!1,
 t1 = Thread!val!0,
 l1 = Lock!val!0,
 t2 = Thread!val!1,
 holds = [else -> True],
 waits = [else -> True]]


Раздел 1.

Задача 2. Функция шаринга дома в приложении умного дома. Может ли гость дома выполнять опасные операции

In [8]:
User = DeclareSort('User')
Home = DeclareSort('Home')

role = Function('role', User, StringSort())
hasControl = Function('hasControl', User, Home, BoolSort())

u = Const('u', User)
h = Const('h', Home)

Owner = StringVal("Owner")
Guest = StringVal("Guest")

DeleteDevices = Const('DeleteDevices', Home)

s = Solver()

s.add(ForAll([u, h],
    Implies(role(u) == Owner, hasControl(u, h))
))

s.add(ForAll([u],
    Implies(role(u) == Guest, Not(hasControl(u, DeleteDevices)))
))

print(s.check())
print(s.model())

sat
[DeleteDevices = Home!val!0,
 role = [else -> "!0!"],
 hasControl = [else -> True]]


**Раздел 2. **

Задача 1. Создание массива и обновление значения

In [5]:
A = Array('A', IntSort(), IntSort())
B = Store(A, 3, 10)

s = Solver()

s.add(Select(B, 3) == 10)
s.add(Select(B, 4) == Select(A, 4))

print(s.check())
print(s.model())

sat
[]


**Раздел 2. **

Задача 2. Эмулятор последовательной памяти через массив

In [12]:
Mem = Array('Mem', IntSort(), IntSort())

Mem1 = Store(Mem, 40, 34)

s = Solver()
s.add(Select(Mem1, 40) == 34)

print(s.check())
print(s.model())

sat
[]


In [ ]:
Mem = Array('Mem', IntSort(), IntSort())

Mem1 = Store(Mem, 40, 34)

s = Solver()
s.add(Select(Mem1, 40) == 45)

print(s.check())
print(s.model())

**Раздел 3.**

Задача 1. Проверить, что после инициализации массива все его элементы одинаковы.

In [13]:
Init = Array('Init', IntSort(), IntSort())
i = Int('i')

s = Solver()

s.add(ForAll(i, Select(Init, i) == 0))

print(s.check())
print(s.model())

sat
[Init = K(Int, 0)]


**Раздел 3.**

Задача 2. Проверить, что операции записи элемента в массив затрагивает только значение по указанному индексу

In [14]:
A = Array('A', IntSort(), IntSort())
k = Int('k')
v = Int('v')
i = Int('i')

B = Store(A, k, v)

s = Solver()

s.add(ForAll(i, Implies(i != k, Select(B, i) == Select(A, i))))

print(s.check())
print(s.model())

sat
[k = 3,
 A = Store(K(Int, 2), 3, 5),
 v = 4,
 Ext = [else -> 3]]


Раздел 4. Оптимизация.

Задача 1. Найти максимальное значение линейного уравнения 5x + 3y + 4z
Ограничения:
- 0 <= x <= 15
- 0 <= y <= 10
- 0 <= z <= 12
- 2x + y + z <= 25
- x + 3y + 2z <= 30
- x + y >= 5

In [16]:
x, y, z = Ints('x y z')

opt = Optimize()

opt.add(x >= 0, x <= 15)
opt.add(y >= 0, y <= 10)
opt.add(z >= 0, z <= 12)

opt.add(2*x + y + z <= 25)
opt.add(x + 3*y + 2*z <= 30)
opt.add(x + y >= 5)

objective = 5*x + 3*y + 4*z

opt.maximize(objective)

opt.check()
m = opt.model()

print(m)
print("max value =", m.eval(objective))

[x = 7, y = 0, z = 11]
max value = 79


Раздел 4.

Задача 2. Найти ближайшую к заданной точку

In [18]:
x = Int('x')

opt = Optimize()

opt.add(x >= -100, x <= 100)

distance = Abs(x - 29)

opt.minimize(distance)

opt.check()
m = opt.model()

print(m)
print("distance =", m.eval(distance))

[x = 29]
distance = 0


**Раздел 5. Множественные солверы**

Задача 1. Копирование ограничений

In [19]:
x, y = Ints('x y')

s1 = Solver()
s1.add(x > 10, y > 10)

print("s1:", s1)

s2 = Solver()
print("s2:", s2)

s2.add(s1.assertions())

print("s2 после копирования:", s2)

print(s2.check())
print(s2.model())

s1: [x > 10, y > 10]
s2: []
s2 после копирования: [x > 10, y > 10]
sat
[y = 11, x = 11]


Раздел 5. Задача 2. Добавление новых ограничений в другой солвер

In [20]:
x, y = Ints('x y')

s1 = Solver()
s1.add(x >= 0, y >= 0)

s2 = Solver()
s2.add(s1.assertions())

s2.add(x + y == 10)

print(s2.check())
print(s2.model())

sat
[y = 0, x = 10]
